# Chapter 2 - Tokens and Token Embeddings
*Exploring tokens and embeddings as an integral part of building LLMs*

『핸즈온 LLM』 2장 실습 노트북

> 출처: [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/tree/main/chapter02) (Apache-2.0)  
> 공식 노트북을 가져와 주석을 덧붙였습니다.

### [OPTIONAL] - Installing Packages on Colab

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [ ]:
# Colab에서 돌릴 때만 주석 풀고 실행
# 버전이 예전 걸로 꽤 고정돼 있어서, 설치하다 막히면 버전 표기부터 의심해볼 것
# %%capture
# !pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

# Downloading and Running An LLM

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately and keep them as such so that we can explore them separately.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1장이랑 같은 Phi-3 모델
# 이번엔 pipeline 없이 토크나이저를 직접 만져볼 거라서 둘 다 변수로 들고 있음
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

In [ ]:
# 끝에 붙은 <|assistant|>는 "이제 네가 답할 차례"라는 뜻의 Phi-3 특수 토큰
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# 문자열 → 토큰 id 텐서로 바꾸고 GPU로 옮기기
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# 20토큰만 짧게 생성해봄
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=20
)

# 다시 사람이 읽을 수 있는 문자열로 디코딩
print(tokenizer.decode(generation_output[0]))

In [ ]:
# 모델이 실제로 보는 건 이 숫자들. 맨 앞의 1은 문장 시작 토큰 <s>
print(input_ids)

In [ ]:
# id를 하나씩 디코딩해보면 단어가 어디서 잘리는지 보임 (apolog + izing, trag + ic 같은 식)
for id in input_ids[0]:
   print(tokenizer.decode(id))

In [ ]:
# 입력 토큰 뒤에 새로 생성된 토큰들이 그대로 이어 붙어 있음
generation_output

In [ ]:
# 3323은 "Sub", 622는 "ject"인데 같이 디코딩하면 "Subject"
# 단어 하나가 토큰 여러 개로 나뉠 수 있다는 걸 확인하는 부분
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

# Comparing Trained LLM Tokenizers


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 토큰마다 배경색을 번갈아 칠해서 어디서 잘렸는지 한눈에 보이게 하는 함수
colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        # \x1b[...m 은 터미널 색상 코드라서 출력에서는 배경색으로 보임
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

In [ ]:
# 대소문자, 이모지, 한자, 코드, 탭 공백, 숫자를 일부러 섞어 놓은 테스트 문장
# 토크나이저마다 뭘 잘 처리하고 뭘 못 하는지 비교하기 좋음
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

In [ ]:
# BERT uncased: 전부 소문자로 바꾸고, 이모지나 한자처럼 모르는 건 [UNK]로 날려버림
show_tokens(text, "bert-base-uncased")

In [ ]:
# BERT cased: 대소문자는 살아 있는데 CAPITALIZATION이 엄청 잘게 쪼개짐
show_tokens(text, "bert-base-cased")

In [ ]:
# GPT-2: 앞 공백까지 토큰에 포함되고, 이모지·한자는 바이트 단위로 쪼개져서 깨진 문자로 보임
show_tokens(text, "gpt2")

In [ ]:
# Flan-T5: 줄바꿈과 공백 정보가 사라지고, 이모지·한자는 <unk>
show_tokens(text, "google/flan-t5-small")

In [ ]:
# GPT-4 토크나이저. 원래는 tiktoken인데 HF에 올라와 있는 같은 토크나이저를 씀
# 공백 여러 개를 한 토큰으로 묶고, elif 같은 코드 키워드도 통째로 처리함
show_tokens(text, "Xenova/gpt-4")

In [ ]:
# 코드 특화 모델 StarCoder2. HF에서 사용 신청을 해야 불러올 수 있음
# 숫자를 한 자리씩 쪼개는 게 눈에 띔
show_tokens(text, "bigcode/starcoder2-15b")

In [ ]:
# Galactica: 과학 논문용 모델이라 그런지 숫자를 한 자리씩 나눔
show_tokens(text, "facebook/galactica-1.3b")

In [ ]:
# 1장에서 쓴 Phi-3. Llama 2 토크나이저 기반이라 숫자를 한 자리씩 쪼갬
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

# Contextualized Word Embeddings From a Language Model (Like BERT)

In [ ]:
from transformers import AutoModel, AutoTokenizer

# 토크나이저는 deberta-base, 모델은 deberta-v3-xsmall (책 코드 그대로)
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# 다음 토큰 예측 헤드 없이 마지막 은닉 상태만 꺼내주는 모델
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

tokens = tokenizer('Hello world', return_tensors='pt')

# [0]이 last_hidden_state, 즉 토큰별 문맥 임베딩
output = model(**tokens)[0]

In [ ]:
# (배치 1, 토큰 4개, 임베딩 384차원)
output.shape

In [ ]:
# 토큰 4개 = [CLS] Hello world [SEP]
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

In [ ]:
# 같은 단어라도 앞뒤 문장에 따라 이 벡터가 달라짐 → 문맥화된 임베딩
output

# Text Embeddings (For Sentences and Whole Documents)

In [ ]:
from sentence_transformers import SentenceTransformer

# 문장 전체를 벡터 하나로 바꿔주는 모델
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

vector = model.encode("Best movie ever!")

In [ ]:
# 토큰별이 아니라 문장 하나에 벡터 하나라서 (768,)
vector.shape

# Word Embeddings Beyond LLMs


In [ ]:
import gensim.downloader as api

# 위키피디아로 학습된 GloVe 50차원 임베딩 (66MB 정도 받음)
# 더 큰 걸 쓰고 싶으면 "word2vec-google-news-300" 같은 것도 있음
# 전체 목록: https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

In [ ]:
# king과 가까운 단어 11개. 첫 번째는 자기 자신이라 유사도가 1.0
model.most_similar([model['king']], topn=11)

# Recommending songs by embeddings

In [ ]:
import pandas as pd
from urllib import request

# 플레이리스트 데이터: 한 줄에 플레이리스트 하나, 노래 id가 공백으로 나열돼 있음
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# 앞의 두 줄은 메타데이터라 건너뜀
lines = data.read().decode("utf-8").split('\n')[2:]

# 노래가 하나뿐인 플레이리스트는 같이 나오는 노래가 없어서 학습에 쓸모가 없으니 제외
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# 노래 id → 제목, 아티스트 표
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [ ]:
# 플레이리스트는 그냥 노래 id 나열
# 플레이리스트를 "문장", 노래를 "단어"처럼 취급하는 게 이 실습의 핵심
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

In [ ]:
from gensim.models import Word2Vec

# 같은 플레이리스트에 자주 같이 나오는 노래끼리 가까워지도록 임베딩 학습
# window=20: 플레이리스트가 길어서 앞뒤로 넓게 봄
# negative=50: 네거티브 샘플을 넉넉하게 뽑음
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

In [ ]:
song_id = 2172

# 2172번 노래와 임베딩이 가장 비슷한 노래 10개
model.wv.most_similar(positive=str(song_id))

In [ ]:
# 2172번이 무슨 노래인지 확인
print(songs_df.iloc[2172])

In [ ]:
import numpy as np

def print_recommendations(song_id):
    # 비슷한 노래 5개의 id만 뽑아서
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    # 제목, 아티스트로 바꿔서 보여줌
    return  songs_df.iloc[similar_songs]

# 메탈 곡을 넣으면 메탈·하드록 계열이 추천됨
print_recommendations(2172)

In [ ]:
# 같은 결과가 나오는지 한 번 더 확인
print_recommendations(2172)

In [ ]:
# 힙합 곡을 넣으면 힙합 곡들이 나옴
# 장르 정보는 하나도 안 줬는데 플레이리스트만으로 이런 게 학습된다는 게 신기한 부분
print_recommendations(842)